# Explore WordNet with NLTK

This notebook helps you inspect **WordNet** entries for any English word (with a focus on nouns).

## What is WordNet?
WordNet is a large lexical database of English where words are grouped into sets of cognitive synonyms called **synsets**.

## What is a synset?
A **synset** is a specific sense of a word (for example, one sense of `bank` means a financial institution, another means land by a river).

## What this notebook displays
For each synset found for a word, this notebook shows:
- Synset name
- Part of speech
- Definition (gloss)
- Example sentences
- Lemmas (synonyms)
- Hypernyms and hyponyms
- Holonyms and meronyms (member/part/substance)
- Attributes, entailments, and `similar_tos` (when available)

It also includes optional advanced helpers to inspect hypernym paths and compare two words side by side.

In [1]:
# Setup cell: ensure NLTK and WordNet resources are available.
# This cell is robust: it installs/downloads only when needed.

import sys
import subprocess
import importlib

def _ensure_package(package_name):
    """Install a Python package with pip if it is not already importable."""
    try:
        importlib.import_module(package_name)
        print(f"Package '{package_name}' is already installed.")
    except ImportError:
        print(f"Package '{package_name}' not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"Installed '{package_name}'.")

# Ensure NLTK itself is installed.
_ensure_package("nltk")

import nltk
from nltk.corpus import wordnet as wn

def _ensure_nltk_resource(resource_path, download_name):
    """Download an NLTK corpus/model if missing."""
    try:
        nltk.data.find(resource_path)
        print(f"NLTK resource '{download_name}' is available.")
    except LookupError:
        print(f"NLTK resource '{download_name}' missing. Downloading...")
        nltk.download(download_name)

# WordNet corpus and multilingual index used by many WordNet installs.
_ensure_nltk_resource("corpora/wordnet", "wordnet")
_ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

print("Setup complete. Ready to inspect WordNet.")

Package 'nltk' is already installed.
NLTK resource 'wordnet' missing. Downloading...
NLTK resource 'omw-1.4' missing. Downloading...
Setup complete. Ready to inspect WordNet.


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\xiaoy\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\xiaoy\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Core inspection utilities.
# Main entry point: inspect_wordnet(word, pos=None, noun_only_words=False, exact_match_only=False)

from typing import List, Optional, Tuple

# Human-friendly labels and WordNet POS mappings.
POS_MAP = {
    None: None,
    "noun": wn.NOUN,
    "verb": wn.VERB,
    "adjective": wn.ADJ,
    "adverb": wn.ADV,
}

POS_LABEL = {
    "n": "noun",
    "v": "verb",
    "a": "adjective",
    "s": "adjective satellite",
    "r": "adverb",
}

def _clean_list(values: List[str], empty_text: str = "(none)") -> str:
    """Return a readable comma-separated string, even for empty lists."""
    if not values:
        return empty_text
    return ", ".join(values)

def _format_synset_name(name: str) -> str:
    """Convert a WordNet synset id like 'bank.n.01' to a readable lemma name."""
    return name.split(".", 1)[0].replace("_", " ")

def _synset_items(synsets) -> List[str]:
    """Format related synsets as 'name - definition'."""
    items = []
    for s in synsets:
        try:
            label = _format_synset_name(s.name())
            definition = s.definition()
            items.append(f"{label} - {definition}" if definition else label)
        except Exception:
            # Fallback if any individual synset is malformed.
            items.append(str(s))
    return items

def _print_section(title: str, values: List[str]):
    """Consistent section rendering for notebook readability."""
    print(f"  {title}:")
    if not values:
        print("    (none)")
        return
    for item in values:
        print(f"    - {item}")

def _normalize_lookup(text: str) -> str:
    """Normalize user input and lemma names for exact matching."""
    return text.replace("_", " ").strip().lower()

def _synset_head_word(syn) -> str:
    """Return the canonical word form used by the synset name."""
    return _format_synset_name(syn.name())

def _filter_synsets(
    synsets,
    word: str,
    noun_only_words: bool,
    exact_match_only: bool,
) -> List[Tuple[object, List[str]]]:
    """Filter synsets by POS and canonical word constraints."""
    if not noun_only_words and not exact_match_only:
        return [
            (syn, [lemma.name().replace("_", " ") for lemma in syn.lemmas()])
            for syn in synsets
        ]

    normalized_word = _normalize_lookup(word)
    filtered_synsets = []

    for syn in synsets:
        if noun_only_words and syn.pos() != wn.NOUN:
            continue

        lemma_names = [lemma.name().replace("_", " ") for lemma in syn.lemmas()]
        normalized_lemmas = {_normalize_lookup(lemma.name()) for lemma in syn.lemmas()}

        if exact_match_only:
            if _normalize_lookup(_synset_head_word(syn)) != normalized_word:
                continue
            if normalized_word not in normalized_lemmas:
                continue

        if lemma_names:
            filtered_synsets.append((syn, lemma_names))

    return filtered_synsets

def _ensure_sentence(text: str) -> str:
    """Ensure sentence-like output ends with terminal punctuation."""
    text = text.strip()
    if not text:
        return ""
    if text[-1] not in ".!?":
        text += "."
    return text

def _looks_like_plural_noun(text: str) -> bool:
    """Use a light heuristic to avoid adding singular articles to plural nouns."""
    token = text.strip().split()[-1].lower() if text.strip() else ""
    if not token:
        return False
    if token in {"news"}:
        return False
    return token.endswith("s") and not token.endswith(("ss", "us", "is"))

def _looks_like_proper_noun(text: str) -> bool:
    """Keep user-supplied capitalization for names, acronyms, and titles."""
    letters = [ch for ch in text if ch.isalpha()]
    return any(ch.isupper() for ch in letters) and text != text.lower()

def _indefinite_article(text: str) -> str:
    """Choose a simple indefinite article for common noun summaries."""
    token = text.strip().split()[0].lower() if text.strip() else ""
    if not token:
        return "A"
    if token.startswith(("honest", "honor", "hour", "heir")):
        return "An"
    if token.startswith(("one", "uni", "use", "user", "euro", "ufo")):
        return "A"
    return "An" if token[0] in "aeiou" else "A"

def _definition_subject(word: str, pos: str) -> Tuple[str, str]:
    """Choose a more natural subject phrase and verb for the summary sentence."""
    word_text = word.strip()
    if pos == wn.VERB:
        return f"To {word_text}", "is to"
    if pos != wn.NOUN:
        return word_text, "means"
    if _looks_like_plural_noun(word_text) or _looks_like_proper_noun(word_text):
        return word_text, "are" if _looks_like_plural_noun(word_text) else "is"
    return f"{_indefinite_article(word_text)} {word_text}", "is"

def _definition_sentence(word: str, pos: str, definition: str) -> str:
    """Turn a raw WordNet definition into a sentence that names the word."""
    definition_text = (definition or "(none)").strip()
    subject, linking_verb = _definition_subject(word, pos)
    if not definition_text or definition_text == "(none)":
        return _ensure_sentence(f"{subject} has no WordNet definition")
    return _ensure_sentence(f"{subject} {linking_verb} {definition_text}")

def _synset_sections(syn, lemmas: List[str]) -> List[Tuple[str, str, List[str]]]:
    """Return inspectable sections with display labels and sentence labels."""
    similar_tos = syn.similar_tos() if hasattr(syn, "similar_tos") else []
    return [
        ("Example sentences", "Example sentences", syn.examples() or []),
        ("Lemma names (synonyms)", "Lemma names", lemmas),
        ("Hypernyms", "Hypernyms", _synset_items(syn.hypernyms())),
        ("Hyponyms", "Hyponyms", _synset_items(syn.hyponyms())),
        ("Member holonyms", "Member holonyms", _synset_items(syn.member_holonyms())),
        ("Part holonyms", "Part holonyms", _synset_items(syn.part_holonyms())),
        ("Substance holonyms", "Substance holonyms", _synset_items(syn.substance_holonyms())),
        ("Member meronyms", "Member meronyms", _synset_items(syn.member_meronyms())),
        ("Part meronyms", "Part meronyms", _synset_items(syn.part_meronyms())),
        ("Substance meronyms", "Substance meronyms", _synset_items(syn.substance_meronyms())),
        ("Attributes", "Attributes", _synset_items(syn.attributes())),
        ("Entailments", "Entailments", _synset_items(syn.entailments())),
        ("Similar_tos", "Similar tos", _synset_items(similar_tos)),
    ]

def _section_sentence(label: str, values: List[str]) -> Optional[str]:
    """Convert a non-empty inspect_wordnet section into one readable sentence."""
    cleaned_values = [
        value.strip()
        for value in values
        if isinstance(value, str) and value.strip() and value.strip() != "(none)"
    ]
    if not cleaned_values:
        return None

    joiner = ", " if label == "Lemma names" else "; "
    if label == "Example sentences":
        return _ensure_sentence(f"{label} include {joiner.join(cleaned_values)}")
    return _ensure_sentence(f"{label} are {joiner.join(cleaned_values)}")

def _collect_synset_summary(syn, lemmas: List[str], include_related_info: bool = False) -> str:
    """Build a sentence-level summary for one filtered synset."""
    subject_word = _format_synset_name(syn.name())
    sentences = [
        _definition_sentence(subject_word, syn.pos(), syn.definition()),
    ]

    if not include_related_info:
        return " ".join(sentences)

    for _, sentence_label, values in _synset_sections(syn, lemmas):
        section_sentence = _section_sentence(sentence_label, values)
        if section_sentence:
            sentences.append(section_sentence)

    return " ".join(sentences)

def collect_wordnet_definitions(
    word: str,
    pos: Optional[str] = None,
    noun_only_words: bool = True,
    exact_match_only: bool = True,
    include_related_info: bool = False,
) -> List[str]:
    """Return WordNet definitions, optionally expanded with inspect_wordnet relation details."""
    if not isinstance(word, str) or not word.strip():
        print("Please provide a non-empty word string.")
        return []

    normalized_word = word.strip()
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None

    if normalized_pos not in POS_MAP:
        valid = ["noun", "verb", "adjective", "adverb", "None"]
        print(f"Invalid pos='{pos}'. Valid options: {', '.join(valid)}")
        return []

    wn_pos = POS_MAP[normalized_pos]
    raw_synsets = wn.synsets(normalized_word, pos=wn_pos) if wn_pos else wn.synsets(normalized_word)
    synsets = _filter_synsets(raw_synsets, normalized_word, noun_only_words, exact_match_only)
    return [
        _collect_synset_summary(syn, lemmas, include_related_info=include_related_info)
        for syn, lemmas in synsets
    ]

def inspect_wordnet(
    word: str,
    pos: Optional[str] = None,
    noun_only_words: bool = False,
    exact_match_only: bool = False,
):
    """
    Inspect WordNet synsets for a given word.

    Parameters
    ----------
    word : str
        Input word to inspect.
    pos : str | None
        Optional POS filter: noun, verb, adjective, adverb, or None.
    noun_only_words : bool
        If True, keep only noun synsets in the returned results.
    exact_match_only : bool
        If True, keep only synsets whose canonical word exactly matches the input word.
    """
    if not isinstance(word, str) or not word.strip():
        print("Please provide a non-empty word string.")
        return

    normalized_word = word.strip()
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None

    if normalized_pos not in POS_MAP:
        valid = ["noun", "verb", "adjective", "adverb", "None"]
        print(f"Invalid pos='{pos}'. Valid options: {', '.join(valid)}")
        return

    wn_pos = POS_MAP[normalized_pos]

    # Retrieve synsets with optional POS filtering.
    raw_synsets = wn.synsets(normalized_word, pos=wn_pos) if wn_pos else wn.synsets(normalized_word)
    synsets = _filter_synsets(raw_synsets, normalized_word, noun_only_words, exact_match_only)

    print("=" * 90)
    print(f"WordNet inspection for word: '{normalized_word}'")
    print(f"POS filter: {normalized_pos if normalized_pos else 'None (all parts of speech)'}")
    print(f"Noun-only synsets: {noun_only_words}")
    print(f"Exact lemma match: {exact_match_only}")
    print(f"Synsets found: {len(synsets)}")
    print("=" * 90)

    # Robust handling when no synsets exist after filtering.
    if not synsets:
        print("No synsets found for this word with the selected filters.")
        return

    for idx, (syn, lemmas) in enumerate(synsets, start=1):
        print(f"\n[{idx}] {syn.name()}")
        print("-" * 90)

        # Basic synset fields.
        pos_label = POS_LABEL.get(syn.pos(), syn.pos())
        print(f"  Synset name : {syn.name()}")
        print(f"  POS         : {pos_label}")
        print(f"  Definition  : {syn.definition() if syn.definition() else '(none)'}")

        for section_title, _, values in _synset_sections(syn, lemmas):
            _print_section(section_title, values)

        print("-" * 90)

    print("\nInspection complete.")


## Optional Advanced Helpers

This section includes:
1. A helper to print hypernym path tree(s) for a selected synset.
2. A helper to compare two words by listing their synsets side by side.

In [3]:
# Advanced helper functions.

import textwrap

def print_hypernym_paths_for_word(word: str, pos: Optional[str] = None, synset_index: int = 0):
    """
    Print all hypernym paths for one selected synset of a word.

    Parameters
    ----------
    word : str
        Target word.
    pos : str | None
        Optional POS filter: noun, verb, adjective, adverb, or None.
    synset_index : int
        Zero-based index of the synset to inspect.
    """
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None
    if normalized_pos not in POS_MAP:
        print(f"Invalid pos='{pos}'.")
        return

    wn_pos = POS_MAP[normalized_pos]
    synsets = wn.synsets(word, pos=wn_pos) if wn_pos else wn.synsets(word)

    if not synsets:
        print(f"No synsets found for '{word}'.")
        return

    if synset_index < 0 or synset_index >= len(synsets):
        print(f"synset_index out of range. Choose 0 to {len(synsets)-1}.")
        return

    target = synsets[synset_index]
    paths = target.hypernym_paths() or []

    print("=" * 90)
    print(f"Hypernym paths for: {target.name()} — {target.definition()}")
    print(f"Total paths: {len(paths)}")
    print("=" * 90)

    if not paths:
        print("No hypernym paths available for this synset.")
        return

    for i, path in enumerate(paths, start=1):
        print(f"\nPath {i}:")
        for depth, node in enumerate(path):
            indent = "  " * depth
            print(f"{indent}- {node.name()} ({POS_LABEL.get(node.pos(), node.pos())})")

def compare_words_synsets(word1: str, word2: str, pos: Optional[str] = None, max_synsets: int = 8):
    """
    Compare synsets of two words side by side in plain text.
    """
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None
    if normalized_pos not in POS_MAP:
        print(f"Invalid pos='{pos}'.")
        return

    wn_pos = POS_MAP[normalized_pos]
    synsets1 = wn.synsets(word1, pos=wn_pos) if wn_pos else wn.synsets(word1)
    synsets2 = wn.synsets(word2, pos=wn_pos) if wn_pos else wn.synsets(word2)

    print("=" * 120)
    print(f"Synset comparison: '{word1}' vs '{word2}' | POS: {normalized_pos if normalized_pos else 'all'}")
    print(f"Counts: {len(synsets1)} vs {len(synsets2)}")
    print("=" * 120)

    if not synsets1 and not synsets2:
        print("No synsets found for either word.")
        return

    width = 58
    rows = max(min(len(synsets1), max_synsets), min(len(synsets2), max_synsets), 1)

    def _syn_line(s):
        if s is None:
            return "(none)"
        return f"{s.name()} — {s.definition()}"

    header_left = f"{word1} synsets"
    header_right = f"{word2} synsets"
    print(f"{header_left:<{width}} | {header_right:<{width}}")
    print("-" * width + "-+-" + "-" * width)

    for i in range(rows):
        s1 = synsets1[i] if i < len(synsets1) and i < max_synsets else None
        s2 = synsets2[i] if i < len(synsets2) and i < max_synsets else None

        left = textwrap.shorten(_syn_line(s1), width=width, placeholder="...")
        right = textwrap.shorten(_syn_line(s2), width=width, placeholder="...")
        print(f"{left:<{width}} | {right:<{width}}")

    if len(synsets1) > max_synsets or len(synsets2) > max_synsets:
        print("\n(Comparison truncated by max_synsets.)")

## Demo Cells

In [4]:
# Demo: director
inspect_wordnet("director")

WordNet inspection for word: 'director'
POS filter: None (all parts of speech)
Noun-only synsets: False
Exact lemma match: False
Synsets found: 5

[1] director.n.01
------------------------------------------------------------------------------------------
  Synset name : director.n.01
  POS         : noun
  Definition  : someone who controls resources and expenditures
  Example sentences:
    (none)
  Lemma names (synonyms):
    - director
    - manager
    - managing director
  Hypernyms:
    - administrator - someone who administers a business
  Hyponyms:
    - manageress - a woman manager
    - bank manager - manager of a branch office of a bank
    - district manager - a manager who supervises the sales activity for a district
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Similar_tos:
    (none)


In [5]:
# Demo: career
inspect_wordnet("apple")

WordNet inspection for word: 'apple'
POS filter: None (all parts of speech)
Noun-only synsets: False
Exact lemma match: False
Synsets found: 2

[1] apple.n.01
------------------------------------------------------------------------------------------
  Synset name : apple.n.01
  POS         : noun
  Definition  : fruit with red or yellow or green skin and sweet to tart crisp whitish flesh
  Example sentences:
    (none)
  Lemma names (synonyms):
    - apple
  Hypernyms:
    - edible fruit - edible reproductive body of a seed plant especially one having sweet flesh
    - pome - a fleshy fruit (apple or pear or related fruits) having seed chambers and an outer fleshy part
  Hyponyms:
    - eating apple - an apple used primarily for eating raw without cooking
    - crab apple - small sour apple; suitable for preserving
    - cooking apple - an apple used primarily in cooking for pies and applesauce etc
  Member holonyms:
    (none)
  Part holonyms:
    - apple - native Eurasian tree widely

In [6]:
# Demo: bank
inspect_wordnet("us")

WordNet inspection for word: 'us'
POS filter: None (all parts of speech)
Noun-only synsets: False
Exact lemma match: False
Synsets found: 4

[1] united_states.n.01
------------------------------------------------------------------------------------------
  Synset name : united_states.n.01
  POS         : noun
  Definition  : North American republic containing 50 states - 48 conterminous states in North America plus Alaska in northwest North America and the Hawaiian Islands in the Pacific Ocean; achieved independence in 1776
  Example sentences:
    (none)
  Lemma names (synonyms):
    - United States
    - United States of America
    - America
    - the States
    - US
    - U.S.
    - USA
    - U.S.A.
  Hypernyms:
    (none)
  Hyponyms:
    (none)
  Member holonyms:
    - north atlantic treaty organization - an international organization created in 1949 by the North Atlantic Treaty for purposes of collective security
    - organization of american states - an association including mo

In [15]:
# Demo: apple
inspect_wordnet("United_States",noun_only_words=True,exact_match_only=False)

WordNet inspection for word: 'United_States'
POS filter: None (all parts of speech)
Noun-only synsets: True
Exact lemma match: False
Synsets found: 2

[1] united_states.n.01
------------------------------------------------------------------------------------------
  Synset name : united_states.n.01
  POS         : noun
  Definition  : North American republic containing 50 states - 48 conterminous states in North America plus Alaska in northwest North America and the Hawaiian Islands in the Pacific Ocean; achieved independence in 1776
  Example sentences:
    (none)
  Lemma names (synonyms):
    - United States
    - United States of America
    - America
    - the States
    - US
    - U.S.
    - USA
    - U.S.A.
  Hypernyms:
    (none)
  Hyponyms:
    (none)
  Member holonyms:
    - north atlantic treaty organization - an international organization created in 1949 by the North Atlantic Treaty for purposes of collective security
    - organization of american states - an association in

In [12]:
collect_wordnet_definitions("director",noun_only_words=True,exact_match_only=False)

['A director is someone who controls resources and expenditures. Lemma names are director, manager, managing director. Hypernyms are administrator - someone who administers a business. Hyponyms are manageress - a woman manager; bank manager - manager of a branch office of a bank; district manager - a manager who supervises the sales activity for a district.',
 'A director is member of a board of directors. Lemma names are director. Hypernyms are committee member - a member of a committee. Member holonyms are board - a committee having supervisory powers.',
 'A director is someone who supervises the actors and directs the action in the production of a show. Lemma names are director, theater director, theatre director. Hypernyms are supervisor - one who supervises or has charge and direction of. Hyponyms are stage director - someone who supervises the actors and directs the action in the production of a stage show.',
 'A film director is the person who directs the making of a film. Lemma

In [9]:
# Noun-only inspection example (requirement 8).
inspect_wordnet("bank", pos="noun")

WordNet inspection for word: 'bank'
POS filter: noun
Noun-only synsets: False
Exact lemma match: False
Synsets found: 10

[1] bank.n.01
------------------------------------------------------------------------------------------
  Synset name : bank.n.01
  POS         : noun
  Definition  : sloping land (especially the slope beside a body of water)
  Example sentences:
    - they pulled the canoe up on the bank
    - he sat on the bank of the river and watched the currents
  Lemma names (synonyms):
    - bank
  Hypernyms:
    - slope - an elevated geological formation
  Hyponyms:
    - riverbank - the bank of a river
    - waterside - land bordering a body of water
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Similar_tos:
    (none)
---------------------------------------------------------------------

In [10]:
# Advanced demo 1: hypernym path tree for the first noun sense of 'apple'.
print_hypernym_paths_for_word("apple", pos="noun", synset_index=0)

Hypernym paths for: apple.n.01 — fruit with red or yellow or green skin and sweet to tart crisp whitish flesh
Total paths: 3

Path 1:
- entity.n.01 (noun)
  - physical_entity.n.01 (noun)
    - object.n.01 (noun)
      - whole.n.02 (noun)
        - natural_object.n.01 (noun)
          - plant_part.n.01 (noun)
            - plant_organ.n.01 (noun)
              - reproductive_structure.n.01 (noun)
                - fruit.n.01 (noun)
                  - edible_fruit.n.01 (noun)
                    - apple.n.01 (noun)

Path 2:
- entity.n.01 (noun)
  - physical_entity.n.01 (noun)
    - matter.n.03 (noun)
      - solid.n.01 (noun)
        - food.n.02 (noun)
          - produce.n.01 (noun)
            - edible_fruit.n.01 (noun)
              - apple.n.01 (noun)

Path 3:
- entity.n.01 (noun)
  - physical_entity.n.01 (noun)
    - object.n.01 (noun)
      - whole.n.02 (noun)
        - natural_object.n.01 (noun)
          - plant_part.n.01 (noun)
            - plant_organ.n.01 (noun)
            

In [11]:
# Advanced demo 2: side-by-side comparison.
compare_words_synsets("bank", "career", pos=None, max_synsets=6)

Synset comparison: 'bank' vs 'career' | POS: all
Counts: 18 vs 3
bank synsets                                               | career synsets                                            
-----------------------------------------------------------+-----------------------------------------------------------
bank.n.01 — sloping land (especially the slope beside a... | career.n.01 — the particular occupation for which you...  
depository_financial_institution.n.01 — a financial...     | career.n.02 — the general progression of your working...  
bank.n.03 — a long ridge or pile                           | career.v.01 — move headlong at high speed                 
bank.n.04 — an arrangement of similar objects in a row...  | (none)                                                    
bank.n.05 — a supply or stock held in reserve for...       | (none)                                                    
bank.n.06 — the funds held by a gambling house or the...   | (none)                            